# Goal

It is my feeling that the Wigner function of a waveform just traces the $\omega(t)$ field, but its width and amplitude modulated by $\gamma(t)$ and $\xi_f(t)$. Let's see if we can get a shape match.

## Conclusion

I'm pretty convinced that this is the case from a few visual examples. Curiously, the amplitude relationship is not $1:1$, which I had noticed before. Why? I need to answer this because then I can say this equation just gives TF curves given by $\omega(t)$, its strength modulated by $\gamma$. Further notes:

- If $\gamma(t)=\xi_f(t)=0$ and init conditions in position and velocity are e.g. $(1,1)$, the amplitude of the oscillation should NOT die down. If it does, which I observed for higher frequencies, this is a smoking gun for the resolution not being high enough and the discretization effectively acting as damping. This was fixed when I set the resolution to $\texttt{r\_fac}=3$ instead of $1$
- When I did increase the resolution, pycharm ended up using $80\mathrm{GB}$ of memory, crashing. Therefore, it is imperative that you downsample the waveform solution

*Next question: What does the resolution need to be in order to accurately portray a frequency curve?*

In [ ]:
import numpy as np
from jupytext.combine import map_outputs_to_inputs
%matplotlib tk
from phase_III.strain import *
from phase_II.nifty_re_playground.strain_tools import *
from phase_III.strain import *
from phase_III.useful.helpers import get_oscillator_sample
import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)
import nifty.re as jft
key = jax.random.key(42)
import matplotlib.pyplot as plt

In [ ]:
GW150914 = StrainSignalInference(
    key=key,
    event_name="GW150914",
    detector="H1",
    data_duration_of_hdf5_file="4096sec",
    stationarity_time_scale=32,
    e_fac=1,
    r_fac=3,
    alpha_taper_on_data=.1,
    out_name='osc_dlt_later'
)

signal_domain = GW150914.machinery.t_ss
target_domain = GW150914.machinery.t_ds

oscillator_prior_dct = {
    "frequency": {"offset_mean": 1000, "offset_std": (500, 1e-16), "fluctuations": (1, 1), "loglogavgslope": (-6, 1)},  # log fluctuations...
    "damping": {"offset_mean": 0, "offset_std": (5, 1e-16), "fluctuations": (10, 10), "loglogavgslope": (-6, 1)},
    "force": {"offset_mean": 0, "offset_std": (5, 1e-16), "fluctuations": (10, 10), "loglogavgslope": (0, 0.001), },
    "global_amplitude": (1, 1),
    "init_condition": (0., 1),
}

signal_prior = StochasticOscillatorPrior(oscillator_prior_dct, signal_time_domain=signal_domain, localize_force=None, couple_force_to_frequency=False)
oscillator = HarmonicOscillator(signal_domain_times=signal_domain, signal_prior=signal_prior)

In [ ]:

def plot_wigner_function_with_omega_evolution(oscillator_model, key, num=1):
    oscillator = oscillator_model
    t = oscillator.evolution_times

    for _ in range(num):

        # get the sample
        key, waveform_sample, omega_sample, gamma_sample, force_sample = get_oscillator_sample(
            oscillator_model=oscillator,
            key=key
        )

        t = t[::3]
        waveform_sample, omega_sample, gamma_sample, force_sample = waveform_sample[::3], omega_sample[::3], gamma_sample[::3], force_sample[::3]

        print("generated prior samples")

        # compute the stress JFT
        S, t_loc, f_loc = Stress_jft(waveform_sample, time=t)

        fig, axs = plt.subplots(5, 1, figsize=(12, 8), sharex=True, gridspec_kw={'height_ratios':[1,1,1,1,3]})

        # top plot: omega(t)
        axs[0].plot(t, omega_sample, color='blue', lw=2)
        axs[0].set_ylabel(r'$\omega(t)$')
        axs[0].set_title('Omega evolution and Stress TF')
        axs[0].grid(True)

        # middle plot: waveform
        axs[1].plot(t, waveform_sample, color='blue', lw=2)
        axs[1].set_ylabel('$h(t)$')
        axs[1].grid(True)

        # other stuff
        axs[2].plot(t, force_sample, color='blue', lw=2)
        axs[2].set_ylabel(r'$\xi_f(t)$')
        axs[2].grid(True)

        axs[3].plot(t, gamma_sample, color='blue', lw=2)
        axs[3].grid(True)
        axs[3].set_ylabel(r'$\gamma(t)$')

        # bottom plot: TF plane
        visualize_stress(S, cols=t_loc, rows=f_loc, custom_ax=axs[4], delay_plot=True)
        axs[4].set_xlabel('Time')
        axs[4].set_ylabel('Frequency / Stress')

        plt.show(block=True)

In [ ]:
plot_wigner_function_with_omega_evolution(oscillator, key, num=10)

## Wigner function for a simple harmonic oscillator

In [ ]:
oscillator_prior_dct = {
    "frequency": {"offset_mean": 2000, "offset_std": (.1, 1e-16), "fluctuations": (1e-16, 1e-16), "loglogavgslope": (-6, 1)},  # log fluctuations...
    "damping": {"offset_mean": 0, "offset_std": (1e-16, 1e-16), "fluctuations": (1e-16, 1e-16), "loglogavgslope": (-6, 1)},
    "force": {"offset_mean": 0, "offset_std": (1e-16, 1e-16), "fluctuations": (1e-16, 1e-16), "loglogavgslope": (0, 0.001), },
    "global_amplitude": (1, 1e-16),
    "init_condition": (1, 1),
}

signal_prior = StochasticOscillatorPrior(oscillator_prior_dct, signal_time_domain=signal_domain, localize_force=None, couple_force_to_frequency=False)
oscillator = HarmonicOscillator(signal_domain_times=signal_domain, signal_prior=signal_prior)

In [ ]:
plot_wigner_function_with_omega_evolution(oscillator, key, num=1)